In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from constants import CANCER, NORMAL, DATA_DIR, PREPARED_DIR, FEATURES_DIR, WITHOUT_LABEL, WITH_LABEL, WEAK_LABELS
from visualization import plot_labeled_comparison, plot_unlabeled_clusters

# ======================
# Config
# ======================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
EPOCHS = 30
LR = 1e-3
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# ======================
# Dataset
# ======================
class FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# ======================
# CNN head (classification)
# ======================
class CNNClassifier(nn.Module):
    """
    Tête de classification d’un CNN (post-avgpool)
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.net(x)

# ======================
# Load data
# ======================
X_cancer = np.load(FEATURES_DIR / "features_cancer_avgpool.npy")
X_normal = np.load(FEATURES_DIR / "features_normal_avgpool.npy")
X = np.vstack([X_cancer, X_normal])

y_weak = np.load(FEATURES_DIR / "weak_labels_unlabeled.npy")

# Normalisation labels faibles
y_weak = y_weak.reshape(-1)

if set(np.unique(y_weak)).issubset({-1, 1}):
    y_weak = ((y_weak + 1) // 2).astype(float)

# Garder uniquement labels valides
mask = ~np.isnan(y_weak)
X = X[mask]
y = y_weak[mask].astype(int)

print("Samples weak-labeled:", len(y))
print("Distribution:", np.bincount(y))

# ======================
# Train / Val split
# ======================
dataset = FeatureDataset(X, y)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# ======================
# Training
# ======================
model = CNNClassifier(input_dim=X.shape[1]).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            preds = model(xb).argmax(1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(yb.numpy())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Epoch {epoch+1:02d} | Acc={acc:.4f} | F1={f1:.4f}")

# ======================
# Final report
# ======================
print("\nClassification report (weak labels):")
print(classification_report(y_true, y_pred, digits=4))

✅ Loaded unlabeled features: (1406, 2048)
Features shape: (1406, 2048)
Weak labels shape: (1406,)

✅ Samples weak-labeled: 1406
Distribution: [569 837]

🚀 Training on cpu...

Epoch 01 | Acc=0.9716 | F1=0.9760
Epoch 02 | Acc=0.9787 | F1=0.9824
Epoch 03 | Acc=0.9752 | F1=0.9791
Epoch 04 | Acc=0.9823 | F1=0.9852
Epoch 05 | Acc=0.9858 | F1=0.9882
Epoch 06 | Acc=0.9894 | F1=0.9913
Epoch 07 | Acc=0.9681 | F1=0.9730
Epoch 08 | Acc=0.9610 | F1=0.9687
Epoch 09 | Acc=0.9716 | F1=0.9760
Epoch 10 | Acc=0.9752 | F1=0.9797
Epoch 11 | Acc=0.9858 | F1=0.9883
Epoch 12 | Acc=0.9716 | F1=0.9769
Epoch 13 | Acc=0.9823 | F1=0.9853
Epoch 14 | Acc=0.9858 | F1=0.9882
Epoch 15 | Acc=0.9539 | F1=0.9634
Epoch 16 | Acc=0.9894 | F1=0.9912
Epoch 17 | Acc=0.9823 | F1=0.9853
Epoch 18 | Acc=0.9823 | F1=0.9853
Epoch 19 | Acc=0.9823 | F1=0.9853
Epoch 20 | Acc=0.9894 | F1=0.9912
Epoch 21 | Acc=0.9823 | F1=0.9853
Epoch 22 | Acc=0.9823 | F1=0.9852
Epoch 23 | Acc=0.9823 | F1=0.9852
Epoch 24 | Acc=0.9823 | F1=0.9852
Epoch 25 

In [2]:
# ======================
# TEST DE VÉRITÉ : Évaluation sur les Strong Labels
# ======================
print("\n" + "="*50)
print("TEST DE VÉRITÉ : Évaluation sur les STRONG LABELS")
print("="*50)

# 1. Charger les vraies features (Strong Labels)
X_cancer_strong = np.load(FEATURES_DIR / "features_cancer_layer4.npy")
X_normal_strong = np.load(FEATURES_DIR / "features_normal_layer4.npy")

X_strong = np.vstack([X_cancer_strong, X_normal_strong])
y_strong = np.array([1] * len(X_cancer_strong) + [0] * len(X_normal_strong))

# 2. Passer en mode évaluation
model.eval()
X_strong_tensor = torch.tensor(X_strong, dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    outputs = model(X_strong_tensor)
    y_pred_strong = outputs.argmax(1).cpu().numpy()

# 3. Comparer avec la réalité
print(classification_report(y_strong, y_pred_strong, target_names=['Normal', 'Cancer'], digits=4))


TEST DE VÉRITÉ : Évaluation sur les STRONG LABELS
              precision    recall  f1-score   support

      Normal     0.8000    0.9600    0.8727        50
      Cancer     0.9500    0.7600    0.8444        50

    accuracy                         0.8600       100
   macro avg     0.8750    0.8600    0.8586       100
weighted avg     0.8750    0.8600    0.8586       100

